# KΛIROS — MusicGen Audio Generator

**Before running:** Make sure you are using a GPU runtime.
> Runtime → Change runtime type → T4 GPU → Save

Workflow:
1. Run Cell 1 — installs audiocraft (takes ~2 min first time)
2. Run Cell 2 — loads the model (downloads ~1.5GB first time)
3. Paste your prompt in Cell 3 and run it
4. Download the generated WAV from Cell 4

# ─────────────────────────────────────────────────────────────
# KΛIROS — Instagram Reels Clip Cutter
# ─────────────────────────────────────────────────────────────
# Cells 6-7 are independent — no need to run Cells 1-5 first.
#
# Workflow:
# 1. Upload your MP3 (Cell 6)
# 2. Run Cell 7 — generates vertical 9:16 Reel with KΛIROS branding
# 3. Download the MP4 and post to Instagram

In [ ]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers==4.40.0 accelerate scipy soundfile
print('Done. Continue to Cell 2 — no restart needed.')

In [ ]:
# ── CELL 2: Load model ────────────────────────────────────────────────────────
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — switch to GPU runtime!"}')

processor = AutoProcessor.from_pretrained("facebook/musicgen-medium")
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-medium")

if torch.cuda.is_available():
    model = model.to("cuda")

print('Model loaded.')

In [ ]:
# ── CELL 3: Generate audio ────────────────────────────────────────────────────
import torch
import scipy.io.wavfile
import numpy as np

PROMPT   = """A 124 BPM melodic techno track in F minor. Hypnotic, cosmic, and transcendent atmosphere. \
Features shimmering pad washes, rolling four-on-the-floor kick, deep melodic bassline, and cinematic string stabs. \
Inspired by Agents Of Time and Kiasmos. Slow build over 8 bars. \
No lyrics except one whispered phrase buried deep in reverb. Sacred geometry energy — mathematical, meditative, hypnotic."""

TITLE    = "Stasis Form"
DROP     = "morning"
DURATION = 30  # seconds

# ─────────────────────────────────────────────────────────────────────────────
model.set_generation_params = None  # not used with transformers API
inputs = processor(text=[PROMPT], padding=True, return_tensors="pt")

if torch.cuda.is_available():
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

max_tokens = DURATION * 50
print(f'Generating: "{TITLE}" [{DROP}] — {DURATION}s ...')

with torch.inference_mode():
    audio_values = model.generate(**inputs, max_new_tokens=max_tokens)

print('Done.')

In [ ]:
# ── CELL 4: Save and download ─────────────────────────────────────────────────
import scipy.io.wavfile
import numpy as np
import datetime
from google.colab import files

sample_rate = model.config.audio_encoder.sampling_rate
audio_np = audio_values[0, 0].cpu().numpy()
audio_int16 = np.int16(audio_np / np.max(np.abs(audio_np)) * 32767)

date_str   = datetime.date.today().strftime('%Y%m%d')
safe_title = TITLE.lower().replace(' ', '_')
filename   = f"transmission_{date_str}_{DROP}_{safe_title}.wav"

scipy.io.wavfile.write(filename, sample_rate, audio_int16)
print(f'Saved: {filename}')
files.download(filename)
print('Download started.')

In [ ]:
# ── CELL 7: Generate Instagram Reel ──────────────────────────────────────────
# Edit these to match your track
TRACK_TITLE      = "STASIS FORM"         # shown on screen
TRANSMISSION_NUM = "TRANSMISSION 002"    # shown on screen
BPM              = "124 BPM"
KEY              = "F MIN"
CLIP_START       = 30    # seconds into the song to start the clip (skip intros)
CLIP_DURATION    = 30    # seconds — Instagram Reels max 90s, sweet spot is 30s

# ─────────────────────────────────────────────────────────────────────────────
import subprocess, datetime
from google.colab import files

output_file = TRACK_TITLE.lower().replace(' ', '_') + '_reel.mp4'

# Build ffmpeg filter chain:
# - Black 1080x1920 background (9:16 vertical)
# - Red animated waveform in center
# - KΛIROS logo text at top
# - Track name + metadata at bottom
# - Sacred geometry hex grid overlay (subtle)

filter_complex = (
    # Background
    "color=c=0x030305:s=1080x1920:r=30[bg];"

    # Waveform — red, centered vertically
    "[0:a]aformat=channel_layouts=mono,"
    "showwaves=s=1080x300:mode=cline:colors=0xff1a33:scale=sqrt:draw=full[wave];"

    # Dim overlay behind waveform for contrast
    "color=c=0x030305@0.5:s=1080x300[wavedim];"

    # Stack background layers
    "[bg][wavedim]overlay=0:810[bg2];"
    "[bg2][wave]overlay=0:810[bg3];"

    # KΛIROS title — top center
    "[bg3]drawtext="
      "text='K\u039bIROS':"
      "fontsize=120:"
      "fontcolor=0xc0152a:"
      "x=(w-text_w)/2:"
      "y=180:"
      "font=sans-serif:"
      "borderw=2:bordercolor=0xff1a33[t1];"

    # Transmission number
    "[t1]drawtext="
      f"text='{TRANSMISSION_NUM}':"
      "fontsize=32:"
      "fontcolor=0xe8e4dc@0.6:"
      "x=(w-text_w)/2:"
      "y=330:"
      "font=monospace[t2];"

    # Track title
    "[t2]drawtext="
      f"text='{TRACK_TITLE}':"
      "fontsize=72:"
      "fontcolor=0xe8e4dc:"
      "x=(w-text_w)/2:"
      "y=1450:"
      "font=sans-serif[t3];"

    # BPM + Key
    "[t3]drawtext="
      f"text='{BPM}  ·  {KEY}  ·  TECHNO':"
      "fontsize=28:"
      "fontcolor=0xc0152a:"
      "x=(w-text_w)/2:"
      "y=1560:"
      "font=monospace[t4];"

    # Red divider line above track title
    "[t4]drawbox="
      "x=(w-200)/2:y=1420:w=200:h=2:"
      "color=0xc0152a:t=fill[out]"
)

cmd = [
    'ffmpeg', '-y',
    '-ss', str(CLIP_START),
    '-t',  str(CLIP_DURATION),
    '-i',  AUDIO_FILE,
    '-filter_complex', filter_complex,
    '-map', '[out]',
    '-map', '0:a',
    '-c:v', 'libx264', '-preset', 'fast', '-crf', '20',
    '-c:a', 'aac', '-b:a', '192k',
    '-shortest',
    output_file
]

print(f'Cutting clip from {CLIP_START}s, duration {CLIP_DURATION}s...')
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print(f'Done! Downloading {output_file}...')
    files.download(output_file)
else:
    print('Error:')
    print(result.stderr[-2000:])

In [ ]:
# ── CELL 6: Upload your MP3 ───────────────────────────────────────────────────
from google.colab import files

print('Select your MP3 file...')
uploaded = files.upload()
AUDIO_FILE = list(uploaded.keys())[0]
print(f'Uploaded: {AUDIO_FILE}')

In [ ]:
# ── CELL 5: Generate BOTH drops back to back ─────────────────────────────────
import scipy.io.wavfile
import numpy as np
import datetime
from google.colab import files

TRACKS = [
    {
        "drop": "morning",
        "title": "Stasis Form",
        "prompt": """A 124 BPM melodic techno track in F minor. Hypnotic, cosmic, and transcendent atmosphere. \
Features shimmering pad washes, rolling four-on-the-floor kick, deep melodic bassline, and cinematic string stabs. \
Inspired by Agents Of Time and Kiasmos. Slow build over 8 bars. \
No lyrics except one whispered phrase buried deep in reverb. Sacred geometry energy — mathematical, meditative, hypnotic.""",
    },
    {
        "drop": "evening",
        "title": "Cascade Protocol",
        "prompt": """A 134 BPM peak-time industrial techno track in A minor. Relentless, unforgiving, and driving energy. \
Features distorted acid bass, metallic percussion hits, dark atmospheric drone, and punishing kick drum with distorted clap. \
Inspired by Truncate and Industrialyzer. Hard, pounding, relentless. Built for peak-time warehouse raving. \
No breakdowns — pure momentum. Machine precision with human aggression. Dark Berlin underground energy.""",
    },
]

sample_rate = model.config.audio_encoder.sampling_rate
date_str = datetime.date.today().strftime('%Y%m%d')
max_tokens = 30 * 50  # 30 seconds

for t in TRACKS:
    print(f"Generating {t['drop']} drop: '{t['title']}' ...")
    inputs = processor(text=[t['prompt']], padding=True, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.inference_mode():
        audio_values = model.generate(**inputs, max_new_tokens=max_tokens)

    audio_np = audio_values[0, 0].cpu().numpy()
    audio_int16 = np.int16(audio_np / np.max(np.abs(audio_np)) * 32767)

    safe = t['title'].lower().replace(' ', '_')
    fname = f"transmission_{date_str}_{t['drop']}_{safe}.wav"
    scipy.io.wavfile.write(fname, sample_rate, audio_int16)
    files.download(fname)
    print(f'  Downloaded: {fname}')

print('Both tracks done!')